## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

Antes de empezar a resolver las tareas vamos a intentar arreglar el problema del desbalance de clases

**1. Carga y análisis inicial del dataset**
- Cargar datos
- Expandir mensajes
- Analizar distribución de clases
- Identificar desbalance

In [13]:
import pandas as pd
import json

data = pd.read_parquet('data/train_preprocessed.parquet')
df_expanded = data.explode(["messages", "sender_labels", "receiver_labels"])
df_expanded.head()

,messages,sender_labels,receiver_labels,speakers,receivers,absolute_message_index,relative_message_index,seasons,years,game_score,game_score_delta,players,game_id,text_clean,tokens,lemmas
0,Germany!\n\nJust the person I want to speak wi...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,germany just the person i want to speak with i...,"['germany', 'person', 'want', 'speak', 'somewh...","['germany', 'person', 'want', 'speak', 'somewh..."
1,"You've whet my appetite, Italy. What's the sug...",True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,youve whet my appetite italy whats the suggestion,"['ve', 'whet', 'appetite', 'italy', 's', 'sugg...","['ve', 'whet', 'appetite', 'italy', 's', 'sugg..."
2,It seems like there are a lot of ways that cou...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,it seems like there are a lot of ways that cou...,"['like', 'lot', 'ways', 'wrong', 'nt', 'france...","['like', 'lot', 'way', 'wrong', 'not', 'france..."
3,"Yeah, I can’t say I’ve tried it and it works, ...",True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,yeah i can t say i ve tried it and it works ca...,"['yeah', 't', 've', 'tried', 'works', 'cause',...","['yeah', 't', 've', 'try', 'work', 'cause', 'v..."
4,I am just sensing that you don’t like this ide...,True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,i am just sensing that you don t like this ide...,"['sensing', 'don', 't', 'like', 'idea', 'shall...","['sense', 'don', 't', 'like', 'idea', 'shall',..."


**2. Análisis del desbalance de clases**
- Distribución de `sender_labels`
- Distribución de `receiver_labels`

In [10]:
print("Distribución sender_labels:")
print(df_expanded["sender_labels"].value_counts())
print("\nDistribución receiver_labels:")
print(df_expanded["receiver_labels"].value_counts())

Distribución sender_labels:
sender_labels
True     11372
False      522
Name: count, dtype: int64

Distribución receiver_labels:
receiver_labels
True            10390
NOANNOTATION      989
False             515
Name: count, dtype: int64


**3. Corrección del desbalance de clases**
- Uso de *class weights*
- Preparación para entrenamiento

In [14]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -------- sender_labels (binario) --------
y_sender = df_expanded["sender_labels"]

sender_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_sender),
    y=y_sender
)

print("\nClass weights para sender_labels:")
print(sender_class_weights)


# -------- receiver_labels (multiclase) --------
y_receiver = df_expanded["receiver_labels"]

receiver_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_receiver),
    y=y_receiver
)

print("\nClass weights para receiver_labels:")
print(receiver_class_weights)


Class weights para sender_labels:
[11.39272031  0.52295111]

Class weights para receiver_labels:
[7.69838188 4.00876306 0.38158486]


**Resultados**

Durante el entrenamiento de los modelos, utilizaremos estos pesos para que la función de pérdida multiplique el error de cada ejemplo por el peso correspondiente a su clase. De esta manera, el optimizador ajustará los parámetros del modelo para minimizar la pérdida ponderada, lo que mejora significativamente la capacidad del modelo para predecir correctamente las clases minoritarias.

## **Shallow ML**
En esta sección vamos a intentar resolver las tareas con técnicas de shallow ML

### **1. Clasificación binaria**
   

### **2. Predicción del hablante**


## **CNNs o Redes Recurrentes**

En esta sección vamos a entrenar modelos CNN para resolver las tareas

### **1. Clasificación binaria**
   

### **2. Predicción del hablante**
